In [ ]:
#| default_exp fit

In [ ]:
#| export
"""lyca.fit — Model registry, system detection, recommendations, and HF download."""

__all__ = ['MODEL_REGISTRY', 'models', 'syscheck', 'recommend', 'download', 'register_model']

In [ ]:
#| export
import os, re, platform, subprocess, functools
from pathlib import Path

from huggingface_hub import hf_hub_download, HfApi
from fastcore.basics import first
from fastcore.meta import patch


In [ ]:
#| export
MODEL_REGISTRY = [
  # ── Gemma 4 ─────────────────────────────────────────────────────────────
  { 'id':      'gemma4-e2b',
    'repo':    'litert-community/gemma-4-E2B-it-litert-lm',
    'file':    'gemma-4-E2B-it.litertlm',
    'size_gb': 2.58,
    'family':  'gemma4',
    'task':    'chat',
    'params':  '2B',
    'quant':   'mixed-4/8bit',
    'min_ram_gb': 3.5,
    'gpu_ram_gb': 0.8,
    'mac_cpu_tps':  42,
    'mac_gpu_tps': 160,
    'tags':    ['multimodal', 'vision', 'audio'] },

  { 'id':      'gemma4-e4b',
    'repo':    'litert-community/gemma-4-E4B-it-litert-lm',
    'file':    'gemma-4-E4B-it.litertlm',
    'size_gb': 3.65,
    'family':  'gemma4',
    'task':    'chat',
    'params':  '4B',
    'quant':   'mixed-4/8bit',
    'min_ram_gb': 4.0,
    'gpu_ram_gb': 3.2,
    'mac_cpu_tps':  27,
    'mac_gpu_tps': 101,
    'tags':    ['multimodal', 'vision', 'audio', 'recommended'] },

  # ── Gemma 3 ─────────────────────────────────────────────────────────────
  { 'id':      'gemma3-1b',
    'repo':    'litert-community/Gemma3-1B-IT',
    'file':    'gemma3-1b-it-int4.litertlm',
    'size_gb': 0.58,
    'family':  'gemma3',
    'task':    'chat',
    'params':  '1B',
    'quant':   'int4',
    'min_ram_gb': 1.5,
    'gpu_ram_gb': 0.6,
    'mac_cpu_tps': None,
    'mac_gpu_tps': None,
    'tags':    ['fast', 'small'] },

  # ── Phi ─────────────────────────────────────────────────────────────────
  { 'id':      'phi4-mini',
    'repo':    'litert-community/Phi-4-mini-instruct',
    'file':    'Phi-4-mini-instruct_multi-prefill-seq_q8_ekv4096.litertlm',
    'size_gb': 3.91,
    'family':  'phi4',
    'task':    'chat',
    'params':  '3.8B',
    'quant':   'q8',
    'min_ram_gb': 5.0,
    'gpu_ram_gb': 4.0,
    'mac_cpu_tps': None,
    'mac_gpu_tps': None,
    'tags':    ['coding', 'reasoning'] },

  # ── DeepSeek ────────────────────────────────────────────────────────────
  { 'id':      'deepseek-r1-1.5b',
    'repo':    'litert-community/DeepSeek-R1-Distill-Qwen-1.5B',
    'file':    'DeepSeek-R1-Distill-Qwen-1.5B_multi-prefill-seq_q8_ekv4096.litertlm',
    'size_gb': 1.83,
    'family':  'deepseek',
    'task':    'chat',
    'params':  '1.5B',
    'quant':   'q8',
    'min_ram_gb': 2.5,
    'gpu_ram_gb': 2.0,
    'mac_cpu_tps': None,
    'mac_gpu_tps': None,
    'tags':    ['reasoning', 'thinking'] },

  # ── Qwen ────────────────────────────────────────────────────────────────
  { 'id':      'qwen3.5-0.8b',
    'repo':    'litert-community/Qwen3.5-0.8B-LiteRT',
    'file':    'model_multimodal.litertlm',
    'size_gb': 1.16,
    'family':  'qwen3.5',
    'task':    'chat',
    'params':  '0.8B',
    'quant':   'int4',
    'min_ram_gb': 2.0,
    'gpu_ram_gb': 1.2,
    'mac_cpu_tps': None,
    'mac_gpu_tps': None,
    'tags':    ['multimodal', 'small', 'fast'] },

  { 'id':      'qwen3.5-4b',
    'repo':    'litert-community/Qwen3.5-4B-LiteRT',
    'file':    'model_quantized.litertlm',
    'size_gb': 4.27,
    'family':  'qwen3.5',
    'task':    'chat',
    'params':  '4B',
    'quant':   'int4',
    'min_ram_gb': 6.0,
    'gpu_ram_gb': 4.5,
    'mac_cpu_tps': None,
    'mac_gpu_tps': None,
    'tags':    ['multimodal'] },

  { 'id':      'qwen3.5-9b',
    'repo':    'litert-community/Qwen3.5-9B-LiteRT',
    'file':    'model_multimodal.litertlm',
    'size_gb': 9.52,
    'family':  'qwen3.5',
    'task':    'chat',
    'params':  '9B',
    'quant':   'int4',
    'min_ram_gb': 12.0,
    'gpu_ram_gb': 10.0,
    'mac_cpu_tps': None,
    'mac_gpu_tps': None,
    'tags':    ['multimodal', 'large'] },

  # ── SmolLM ──────────────────────────────────────────────────────────────
  { 'id':      'smollm-135m',
    'repo':    'litert-community/SmolLM-135M-Instruct',
    'file':    None,
    'size_gb': None,
    'family':  'smollm',
    'task':    'chat',
    'params':  '135M',
    'quant':   'unknown',
    'min_ram_gb': 1.0,
    'gpu_ram_gb': 0.5,
    'mac_cpu_tps': None,
    'mac_gpu_tps': None,
    'tags':    ['tiny', 'fast'] },
]

In [ ]:
#| export
def register_model(entry: dict) -> None:
    """Add a model entry to MODEL_REGISTRY at runtime.

    Required keys: id, repo, file (may be None), size_gb (may be None), min_ram_gb, task.
    Optional keys: family, params, quant, gpu_ram_gb, mac_cpu_tps, mac_gpu_tps, tags.
    """
    required = {'id', 'repo', 'file', 'size_gb', 'min_ram_gb', 'task'}
    assert required <= set(entry), f'Missing keys: {required - set(entry)}'
    MODEL_REGISTRY.append(entry)

In [ ]:
#| Tests for MODEL_REGISTRY structure and register_model — real data, no mocks
assert len(MODEL_REGISTRY) == 9, f'Expected 9 entries, got {len(MODEL_REGISTRY)}'

required_keys = {'id', 'repo', 'task', 'min_ram_gb'}
for entry in MODEL_REGISTRY:
    assert required_keys <= set(entry), f'Entry {entry.get("id","?")} missing keys: {required_keys - set(entry)}'

ids = [e['id'] for e in MODEL_REGISTRY]
assert len(ids) == len(set(ids)), f'Duplicate IDs found: {[x for x in ids if ids.count(x) > 1]}'

# Test register_model
_test_entry = {'id': '_test_model', 'repo': 'test/repo', 'file': 'test.litertlm',
               'size_gb': 1.0, 'min_ram_gb': 2.0, 'task': 'chat'}
register_model(_test_entry)
assert MODEL_REGISTRY[-1]['id'] == '_test_model'
MODEL_REGISTRY.pop()  # clean up

# Test register_model rejects missing keys
try:
    register_model({'id': 'bad'})
    assert False, 'Should have raised AssertionError'
except AssertionError:
    pass

print('MODEL_REGISTRY and register_model tests passed')

In [ ]:
#| export
def _resolve_entry(model_id: str) -> dict | None:
    "Lookup a MODEL_REGISTRY entry by `id` or by `repo` string."
    return first(MODEL_REGISTRY, lambda e: e['id']==model_id or e['repo']==model_id)


In [ ]:
#| Tests for _resolve_entry — real data, no mocks
e = _resolve_entry('gemma4-e4b')
assert e is not None
assert e['id'] == 'gemma4-e4b'
assert e['repo'] == 'litert-community/gemma-4-E4B-it-litert-lm'

e2 = _resolve_entry('litert-community/gemma-4-E4B-it-litert-lm')
assert e2 is not None
assert e2['id'] == 'gemma4-e4b'

assert _resolve_entry('nonexistent-model') is None
print('_resolve_entry tests passed')

In [ ]:
#| export
def _parse_apple_chip(brand: str) -> tuple:
    "Return (chip_name, tier) from a CPU brand string like 'Apple M4 Max'."
    m = re.search(r'Apple (M\d+)\s*(Pro|Max|Ultra)?', brand)
    if not m: return None, None
    chip = m.group(1)
    tier_map = {None: 1, 'Pro': 2, 'Max': 3, 'Ultra': 4}
    return chip, tier_map[m.group(2)]

In [ ]:
#| Tests for _parse_apple_chip — real strings, no mocks
assert _parse_apple_chip('Apple M4 Max') == ('M4', 3)
assert _parse_apple_chip('Apple M1') == ('M1', 1)
assert _parse_apple_chip('Intel Core i9-12900K') == (None, None)
assert _parse_apple_chip('Apple M2 Pro') == ('M2', 2)
assert _parse_apple_chip('Apple M1 Ultra') == ('M1', 4)
assert _parse_apple_chip('Apple M3') == ('M3', 1)
assert _parse_apple_chip('') == (None, None)
print('_parse_apple_chip tests passed')

In [ ]:
#| export
@functools.lru_cache(maxsize=1)
def syscheck() -> dict:
    "Detect RAM, CPU, platform, GPU availability. Returns a plain dict."
    plat = platform.system()
    arch = platform.machine()
    cpu_brand = 'unknown'
    ram_gb = 0.0
    free_ram_gb = 0.0
    gpu = 'none'
    gpu_vram_gb = None

    if plat == 'Darwin':  # macOS
        plat_name = 'macOS'
        try:
            r = subprocess.run(['sysctl', '-n', 'hw.memsize'],
                               capture_output=True, text=True, timeout=5)
            ram_gb = round(int(r.stdout.strip()) / (1024**3), 1)
        except Exception: pass
        try:
            r = subprocess.run(['sysctl', '-n', 'machdep.cpu.brand_string'],
                               capture_output=True, text=True, timeout=5)
            cpu_brand = r.stdout.strip()
        except Exception: pass
        # Free RAM via vm_stat
        try:
            r = subprocess.run(['vm_stat'], capture_output=True, text=True, timeout=5)
            pages_free = int(re.search(r'Pages free:\s+(\d+)', r.stdout).group(1))
            pages_inactive = int(re.search(r'Pages inactive:\s+(\d+)', r.stdout).group(1))
            free_ram_gb = round((pages_free + pages_inactive) * 4096 / (1024**3), 1)
        except Exception:
            free_ram_gb = round(ram_gb * 0.5, 1)  # rough fallback
        gpu = 'metal' if arch == 'arm64' else 'none'

    elif plat == 'Linux':
        plat_name = 'Linux'
        try:
            with open('/proc/meminfo') as f:
                mem = f.read()
            m_total = re.search(r'MemTotal:\s+(\d+)', mem)
            m_avail = re.search(r'MemAvailable:\s+(\d+)', mem)
            if m_total: ram_gb = round(int(m_total.group(1)) / (1024**2), 1)
            if m_avail: free_ram_gb = round(int(m_avail.group(1)) / (1024**2), 1)
        except Exception: pass
        try:
            with open('/proc/cpuinfo') as f:
                for line in f:
                    if line.startswith('model name'):
                        cpu_brand = line.split(':', 1)[1].strip()
                        break
        except Exception: pass
        # CUDA detection
        try:
            r = subprocess.run(['nvidia-smi', '--query-gpu=memory.total',
                                '--format=csv,noheader,nounits'],
                               capture_output=True, text=True, timeout=5)
            if r.returncode == 0 and r.stdout.strip():
                gpu = 'cuda'
                gpu_vram_gb = round(int(r.stdout.strip().split('\n')[0]) / 1024, 1)
        except Exception: pass

    elif plat == 'Windows':
        plat_name = 'Windows'
        try:
            r = subprocess.run(['wmic', 'ComputerSystem', 'get', 'TotalPhysicalMemory'],
                               capture_output=True, text=True, timeout=5)
            for line in r.stdout.strip().split('\n'):
                line = line.strip()
                if line.isdigit():
                    ram_gb = round(int(line) / (1024**3), 1)
                    break
        except Exception: pass
        # CUDA detection
        try:
            r = subprocess.run(['nvidia-smi', '--query-gpu=memory.total',
                                '--format=csv,noheader,nounits'],
                               capture_output=True, text=True, timeout=5)
            if r.returncode == 0 and r.stdout.strip():
                gpu = 'cuda'
                gpu_vram_gb = round(int(r.stdout.strip().split('\n')[0]) / 1024, 1)
        except Exception: pass
    else:
        plat_name = plat

    # psutil fallback for RAM if we couldn't detect it
    if ram_gb == 0.0:
        try:
            import psutil
            vm = psutil.virtual_memory()
            ram_gb = round(vm.total / (1024**3), 1)
            free_ram_gb = round(vm.available / (1024**3), 1)
        except ImportError: pass

    apple_chip, apple_tier = _parse_apple_chip(cpu_brand)

    return {
        'platform':    plat_name,
        'arch':        arch,
        'cpu':         cpu_brand,
        'ram_gb':      ram_gb,
        'free_ram_gb': free_ram_gb,
        'gpu':         gpu,
        'gpu_vram_gb': gpu_vram_gb,
        'apple_chip':  apple_chip,
        'apple_tier':  apple_tier,
    }

In [ ]:
#| export
def _gpu_usable(spec: dict) -> bool:
    "Return True if GPU backend can be used based on syscheck spec."
    return spec.get('gpu') in ('metal', 'cuda')

In [ ]:
#| Tests for syscheck and _gpu_usable — real system call + synthetic specs, no mocks
spec = syscheck()
expected_keys = {'platform', 'arch', 'cpu', 'ram_gb', 'free_ram_gb', 'gpu', 'gpu_vram_gb', 'apple_chip', 'apple_tier'}
assert expected_keys == set(spec.keys()), f'Keys mismatch: {expected_keys ^ set(spec.keys())}'
assert spec['ram_gb'] > 0, f'RAM should be > 0, got {spec["ram_gb"]}'
assert spec['platform'] in ('macOS', 'Linux', 'Windows'), f'Unexpected platform: {spec["platform"]}'
assert isinstance(spec['arch'], str) and len(spec['arch']) > 0
print(f'syscheck: {spec}')

# _gpu_usable with synthetic specs
assert _gpu_usable({'gpu': 'metal'}) == True
assert _gpu_usable({'gpu': 'cuda'}) == True
assert _gpu_usable({'gpu': 'none'}) == False
assert _gpu_usable({}) == False
print('syscheck and _gpu_usable tests passed')

In [ ]:
#| export
def _fmt_table(entries: list[dict], spec: dict = None) -> str:
    "Format a list of MODEL_REGISTRY entries as an aligned table string."
    has_gpu = _gpu_usable(spec) if spec else False
    header = f"  {'id':<20s} {'params':>6s}  {'size':>6s}  {'RAM req':>7s}  {'GPU?':>4s}  {'est. tps':>8s}  tags"
    sep    = '  ' + '─' * 72
    lines  = [header, sep]
    for e in entries:
        size_str = f"{e['size_gb']:.1f}GB" if e.get('size_gb') is not None else '—'
        ram_str  = f"{e['min_ram_gb']:.1f}GB"
        if has_gpu and e.get('mac_gpu_tps') is not None:
            tps_str = str(e['mac_gpu_tps'])
        elif e.get('mac_cpu_tps') is not None:
            tps_str = str(e['mac_cpu_tps'])
        else:
            tps_str = '—'
        gpu_str = '✓' if has_gpu else '—'
        tags_str = ' '.join(e.get('tags', []))
        lines.append(f"  {e['id']:<20s} {e.get('params',''):>6s}  {size_str:>6s}  {ram_str:>7s}  {gpu_str:>4s}  {tps_str:>8s}  {tags_str}")
    return '\n'.join(lines)

In [ ]:
#| Tests for _fmt_table — real registry entries, no mocks
table = _fmt_table(MODEL_REGISTRY[:3], spec={'gpu': 'metal'})
assert 'id' in table
assert 'params' in table
assert 'size' in table
assert 'RAM req' in table
assert 'gemma4-e2b' in table
assert 'gemma4-e4b' in table
assert 'gemma3-1b' in table

# Test with no GPU
table_no_gpu = _fmt_table(MODEL_REGISTRY[:1], spec={'gpu': 'none'})
assert '—' in table_no_gpu  # GPU column should show —

# Test with empty list
table_empty = _fmt_table([])
assert 'id' in table_empty  # header still present

print('_fmt_table output:')
print(table)
print('\n_fmt_table tests passed')

In [ ]:
#| export
def models(family: str = None, task: str = None, tag: str = None) -> list[dict]:
    "Return (and print) litert-community models from the registry."
    out = MODEL_REGISTRY[:]
    if family: out = [e for e in out if family.lower() in e.get('family', '').lower()]
    if task:   out = [e for e in out if e.get('task') == task]
    if tag:    out = [e for e in out if tag in e.get('tags', [])]
    spec = syscheck()
    print(_fmt_table(out, spec))
    return out

In [ ]:
#| Tests for models — real registry, no mocks
all_models = models()
assert len(all_models) == 9, f'Expected 9 models, got {len(all_models)}'

gemma4 = models(family='gemma4')
assert len(gemma4) == 2
assert all(e['family'] == 'gemma4' for e in gemma4)

reasoning = models(tag='reasoning')
assert len(reasoning) == 2  # phi4-mini and deepseek-r1-1.5b
reasoning_ids = {e['id'] for e in reasoning}
assert 'phi4-mini' in reasoning_ids
assert 'deepseek-r1-1.5b' in reasoning_ids

chat_models = models(task='chat')
assert len(chat_models) == 9  # all models are chat

print('models() tests passed')

In [ ]:
#| export
def recommend(min_tps: float = 5.0, task: str = 'chat', verbose: bool = True) -> list[dict]:
    "Cross-join registry with syscheck, rank models that fit this system."
    spec = syscheck()
    has_gpu = _gpu_usable(spec)
    results = []

    for e in MODEL_REGISTRY:
        # fits_task
        if e.get('task') != task: continue
        # fits_disk: skip unknown sizes
        if e.get('size_gb') is None: continue
        # fits_ram: 15% headroom
        if spec['free_ram_gb'] < e['min_ram_gb'] * 1.15: continue
        # expected_tps
        if has_gpu and e.get('mac_gpu_tps') is not None:
            expected_tps = e['mac_gpu_tps']
        elif e.get('mac_cpu_tps') is not None:
            expected_tps = e['mac_cpu_tps']
        else:
            expected_tps = None
        # meets_speed: None = unknown → include
        if expected_tps is not None and expected_tps < min_tps: continue
        # score: higher is better
        score = (expected_tps or 20) * (1 + 0.1 * len(e.get('tags', [])))
        results.append({**e, '_score': score, '_expected_tps': expected_tps})

    results.sort(key=lambda x: x['_score'], reverse=True)

    if verbose:
        cpu_short = spec.get('apple_chip') or spec.get('cpu', 'unknown')[:20]
        gpu_label = spec['gpu'].title() if spec['gpu'] != 'none' else 'No GPU'
        print(f"lyca: models that fit your system ({spec['ram_gb']} GB RAM, {cpu_short} / {gpu_label})\n")
        print(_fmt_table(results, spec))
        # Find recommended model
        rec = next((e for e in results if 'recommended' in e.get('tags', [])), None)
        if rec:
            print(f'\nUse lyca.download("{rec["id"]}") to fetch the recommended model.')
        elif results:
            print(f'\nUse lyca.download("{results[0]["id"]}") to fetch the top-ranked model.')

    # Clean internal keys before returning
    for r in results:
        r.pop('_score', None)
        r.pop('_expected_tps', None)
    return results

In [ ]:
#| Tests for recommend — real syscheck, no mocks
recs = recommend(verbose=False)
assert isinstance(recs, list)
# Each entry should have an 'id' key
for r in recs:
    assert 'id' in r, f'Entry missing id key: {r}'
# All returned entries should fit current system RAM (with 15% headroom)
spec = syscheck()
for r in recs:
    assert spec['free_ram_gb'] >= r['min_ram_gb'] * 1.15, (
        f"{r['id']} min_ram_gb={r['min_ram_gb']} exceeds free_ram={spec['free_ram_gb']}")
# Internal scoring keys should be cleaned
for r in recs:
    assert '_score' not in r
    assert '_expected_tps' not in r

print(f'recommend returned {len(recs)} models')
# Also test verbose output
recs_verbose = recommend(verbose=True)
print('recommend tests passed')

In [ ]:
#| export
def download(model_id: str, filename: str = None, dest: str = None,
             token: str = None, force: bool = False) -> str:
    """Download a model from HuggingFace. Returns absolute path to .litertlm file.

    `model_id` can be a registry id ('gemma4-e4b') or full HF repo ('litert-community/...').
    """
    entry = _resolve_entry(model_id)
    if entry:
        repo = entry['repo']
        filename = filename or entry.get('file')
    else:
        # Treat model_id as a direct HF repo string
        repo = model_id

    # Auto-discover filename when not specified
    if filename is None:
        files = [f.rfilename for f in HfApi().model_info(repo, files_metadata=False).siblings
                 if f.rfilename.endswith('.litertlm')]
        assert len(files) > 0, f'No .litertlm files found in {repo}'
        filename = sorted(files)[0]  # pick first alphabetically

    dest = dest or str(Path.home() / '.cache' / 'lyca')

    path = hf_hub_download(
        repo_id=repo,
        filename=filename,
        local_dir=dest,
        force_download=force,
        token=token,
    )
    return str(Path(path).resolve())

In [ ]:
#| Tests for download argument resolution — no actual download, no mocks
# Test that _resolve_entry correctly maps registry IDs for download()
e = _resolve_entry('gemma4-e4b')
assert e['repo'] == 'litert-community/gemma-4-E4B-it-litert-lm'
assert e['file'] == 'gemma-4-E4B-it.litertlm'

e2 = _resolve_entry('deepseek-r1-1.5b')
assert e2['repo'] == 'litert-community/DeepSeek-R1-Distill-Qwen-1.5B'
assert e2['file'] == 'DeepSeek-R1-Distill-Qwen-1.5B_multi-prefill-seq_q8_ekv4096.litertlm'

# SmolLM has file=None — download() would auto-discover it
e3 = _resolve_entry('smollm-135m')
assert e3['file'] is None

# Direct repo string — not in registry
assert _resolve_entry('some-org/some-model') is None

print('download argument resolution tests passed')

In [ ]:
#| export
from lyca.core import Chat, AsyncChat

@patch(cls_method=True)
def from_hf(cls: Chat, model_id: str, filename: str = None,
            dest: str = None, token: str = None, **kwargs):
    "Download model from HuggingFace then return a ready Chat."
    path = download(model_id, filename=filename, dest=dest, token=token)
    return cls(path, **kwargs)

@patch(cls_method=True)
def from_hf(cls: AsyncChat, model_id: str, filename: str = None,
            dest: str = None, token: str = None, **kwargs):
    "Download model from HuggingFace then return a ready AsyncChat."
    path = download(model_id, filename=filename, dest=dest, token=token)
    return cls(path, **kwargs)


In [ ]:
#| Tests for Chat.from_hf and AsyncChat.from_hf existence — no mocks
assert hasattr(Chat, 'from_hf'), 'Chat.from_hf not found'
assert callable(Chat.from_hf), 'Chat.from_hf is not callable'
assert hasattr(AsyncChat, 'from_hf'), 'AsyncChat.from_hf not found'
assert callable(AsyncChat.from_hf), 'AsyncChat.from_hf is not callable'
print('Chat.from_hf and AsyncChat.from_hf existence tests passed')